# Lesson 14: Train the decoder language model

In Lesson 13, fixed weights showed the complete forward pass. Now the model reads the shared Tiny Shakespeare corpus and learns by predicting the next character. PyTorch records the operations in a computation graph, calculates gradients with backpropagation, and AdamW updates every model parameter.

The reference tutorial uses a much larger model and thousands of updates. This lesson starts with a small configuration that can train on a local CPU; a GPU is optional. Later, we can increase the context, width, layers, and training time and compare the measured run on a rented GPU.

The original reference is the “Full finished code” cell in [`gpt_dev.ipynb`](../gpt_dev.ipynb). This notebook divides that code into cells so we can inspect data, modules, training curves, and generated text separately. The C and Java lessons show the decoder forward pass in [Lesson 13](../README.md#lesson-13-assemble-a-tiny-decoder-language-model). Their [C](../c/src/bigram_training_main.c) and [Java](../java/src/main/java/learning/karpathy/gpt1/BigramTrainingLesson.java) Lesson 5 implementations derive cross-entropy gradients and apply AdamW for the simpler bigram model; this full Transformer training loop uses PyTorch autograd. Read the official [autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html) and [AdamW API](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html) for the library details.


## Set up the corpus and training batches

We keep the character-level vocabulary from earlier lessons. Each training example is a short window `x`; its target `y` is the same window shifted one character forward. All positions contribute to the cross-entropy loss.


In [ ]:
from pathlib import Path
import time

import torch
import torch.nn as nn
from torch.nn import functional as F

# Keep the first run small enough to understand and run on a CPU.
BATCH_SIZE = 8
BLOCK_SIZE = 16
MAX_ITERS = 100
EVAL_INTERVAL = 20
EVAL_ITERS = 4
LEARNING_RATE = 1e-3
N_EMBD = 16
N_HEAD = 2
N_LAYER = 2
DROPOUT = 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(1337)
torch.set_num_threads(1)  # keep the tiny CPU model from oversubscribing worker threads

data_candidates = (
    Path.cwd() / "gpt-1" / "data" / "input.txt",
    Path.cwd() / "data" / "input.txt",
    Path.cwd().parent / "data" / "input.txt",
)
dataset_path = next((path.resolve() for path in data_candidates if path.is_file()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find gpt-1/data/input.txt. Open the repo root, gpt-1/, or gpt-1/python/.")

text = dataset_path.read_text(encoding="utf-8")
characters = sorted(set(text))
vocab_size = len(characters)
string_to_id = {character: token_id for token_id, character in enumerate(characters)}
id_to_string = {token_id: character for token_id, character in enumerate(characters)}
encode = lambda sample: [string_to_id[character] for character in sample]
decode = lambda token_ids: "".join(id_to_string[token_id] for token_id in token_ids)
all_token_ids = torch.tensor(encode(text), dtype=torch.long)

split_at = int(0.9 * len(all_token_ids))
train_data = all_token_ids[:split_at]
validation_data = all_token_ids[split_at:]


def get_batch(split, generator=None):
    # Pair each context window with the same window shifted by one character.
    source = train_data if split == "train" else validation_data
    starts = torch.randint(len(source) - BLOCK_SIZE, (BATCH_SIZE,), generator=generator)
    inputs = torch.stack([source[start:start + BLOCK_SIZE] for start in starts])
    targets = torch.stack([source[start + 1:start + BLOCK_SIZE + 1] for start in starts])
    return inputs.to(DEVICE), targets.to(DEVICE)


print(f"Dataset: {dataset_path}")
print(f"Characters: {len(text):,}; vocabulary: {vocab_size}; device: {DEVICE}")
print(f"Train/validation tokens: {len(train_data):,}/{len(validation_data):,}")
print(f"Batch shape: [{BATCH_SIZE}, {BLOCK_SIZE}]; model width: {N_EMBD}; layers: {N_LAYER}; heads: {N_HEAD}")


## Build the Transformer modules

These modules match the decoder architecture from Lesson 13: causal attention heads communicate across earlier positions; a feed-forward network transforms each position independently; residual connections and LayerNorm surround both sublayers.


In [ ]:
class CausalAttentionHead(nn.Module):
    # One masked self-attention head.
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        causal_mask = torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        _, time_steps, _ = x.shape
        key = self.key(x)
        query = self.query(x)
        value = self.value(x)
        scores = query @ key.transpose(-2, -1) * key.shape[-1] ** -0.5
        future = ~self.causal_mask[:time_steps, :time_steps]
        scores = scores.masked_fill(future, float("-inf"))
        weights = self.dropout(F.softmax(scores, dim=-1))
        return weights @ value


class MultiHeadAttention(nn.Module):
    def __init__(self, head_count, head_size):
        super().__init__()
        self.heads = nn.ModuleList(CausalAttentionHead(head_size) for _ in range(head_count))
        self.projection = nn.Linear(N_EMBD, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        joined = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.dropout(self.projection(joined))


### Feed-forward network and decoder block

The feed-forward network expands channels fourfold, applies ReLU, and contracts back to model width. Each position goes through the same network independently. The block puts LayerNorm before attention and the feed-forward network and adds both outputs through residual connections.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(N_EMBD, 4 * N_EMBD),
            nn.ReLU(),
            nn.Linear(4 * N_EMBD, N_EMBD),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = N_EMBD // N_HEAD
        self.attention = MultiHeadAttention(N_HEAD, head_size)
        self.feed_forward = FeedForward()
        self.first_norm = nn.LayerNorm(N_EMBD)
        self.second_norm = nn.LayerNorm(N_EMBD)

    def forward(self, x):
        x = x + self.attention(self.first_norm(x))
        x = x + self.feed_forward(self.second_norm(x))
        return x


## Assemble the language model

Token embeddings identify characters; position embeddings identify where they appear in the window. After the blocks, the language-model head returns one score for every possible next character.


In [ ]:
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks = nn.Sequential(*(TransformerBlock() for _ in range(N_LAYER)))
        self.final_norm = nn.LayerNorm(N_EMBD)
        self.language_model_head = nn.Linear(N_EMBD, vocab_size)

    def forward(self, token_ids, targets=None):
        batch_size, time_steps = token_ids.shape
        positions = torch.arange(time_steps, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        x = self.blocks(x)
        logits = self.language_model_head(self.final_norm(x))
        loss = None
        if targets is not None:
            batch, time, vocab = logits.shape
            loss = F.cross_entropy(
                logits.reshape(batch * time, vocab),
                targets.reshape(batch * time),
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, token_ids, new_token_count):
        for _ in range(new_token_count):
            context = token_ids[:, -BLOCK_SIZE:]
            logits, _ = self(context)
            probabilities = F.softmax(logits[:, -1, :], dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1)
            token_ids = torch.cat((token_ids, next_id), dim=1)
        return token_ids


model = TinyGPT().to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Trainable parameters: {parameter_count:,}")


## Step 1 — measure the starting loss

Before learning, evaluate the randomly initialized model on repeatable batches from both data splits. Validation examples are held out: the optimizer never samples them for training.


In [ ]:
@torch.no_grad()
def estimate_loss():
    was_training = model.training
    model.eval()  # dropout is disabled while measuring loss
    losses = {}
    fixed_seeds = {"train": 2025, "validation": 2026}
    for split, seed in fixed_seeds.items():
        generator = torch.Generator().manual_seed(seed)
        batch_losses = []
        for _ in range(EVAL_ITERS):
            inputs, targets = get_batch(split, generator=generator)
            _, loss = model(inputs, targets)
            batch_losses.append(loss.item())
        losses[split] = sum(batch_losses) / len(batch_losses)
    model.train(was_training)
    return losses


initial_losses = estimate_loss()
training_history = [(0, initial_losses["train"], initial_losses["validation"])]
print(f"Initial loss: train {initial_losses['train']:.4f}; validation {initial_losses['validation']:.4f}")
print("Uniform predictions would have loss ln(vocabulary size).")


## Step 2 — backpropagate and update with AdamW

Each iteration samples a new training batch, computes cross-entropy, clears gradients from the previous iteration, calls `loss.backward()` to calculate derivatives through the full Transformer, then calls `optimizer.step()` to update all parameters. Evaluation uses fixed batches and turns dropout off so the curve is easier to compare.

These smaller settings intentionally make this a CPU-sized experiment. Increase the model size or training time after observing this first run; the original reference uses a larger model and a longer schedule.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print("step | train loss | validation loss")
print(f"{0:4d} | {initial_losses['train']:10.4f} | {initial_losses['validation']:15.4f}")

start_time = time.perf_counter()
for step in range(1, MAX_ITERS + 1):
    model.train()
    inputs, targets = get_batch("train")
    _, loss = model(inputs, targets)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_ITERS:
        measured_losses = estimate_loss()
        training_history.append((step, measured_losses["train"], measured_losses["validation"]))
        print(f"{step:4d} | {measured_losses['train']:10.4f} | {measured_losses['validation']:15.4f}")

training_seconds = time.perf_counter() - start_time
final_train_loss = training_history[-1][1]
print(f"Training time: {training_seconds:.1f} seconds on {DEVICE}")
assert final_train_loss < initial_losses["train"], "Training did not lower the fixed training-batch loss."


### Plot the learning curves

Training loss measures examples used for updates. Validation loss measures held-out text. They often move together early on; if training keeps improving while validation worsens, the model may be overfitting.


In [ ]:
import matplotlib.pyplot as plt

steps = [row[0] for row in training_history]
train_losses = [row[1] for row in training_history]
validation_losses = [row[2] for row in training_history]

figure, axis = plt.subplots(figsize=(7, 4))
axis.plot(steps, train_losses, marker="o", label="train")
axis.plot(steps, validation_losses, marker="o", label="validation")
axis.set(title="Tiny GPT training loss", xlabel="training step", ylabel="cross-entropy")
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()
plt.show()


## Step 3 — sample from the trained model

Generation repeatedly crops to the most recent context window, takes the logits at its final position, converts them to probabilities, and samples the next character. With only 100 updates, expect the model to learn some character patterns without producing polished Shakespeare.


In [ ]:
torch.manual_seed(1337)
model.eval()
start_character = "\n" if "\n" in string_to_id else characters[0]
start_ids = torch.tensor([[string_to_id[start_character]]], dtype=torch.long, device=DEVICE)
sampled_ids = model.generate(start_ids, new_token_count=400)[0].tolist()
sample = decode(sampled_ids)

print(f"Start character: {start_character!r}")
print(sample)


### What changed?

The weights now depend on training data instead of fixed formulas. Autograd applied the chain rule through attention, residual connections, LayerNorm, and feed-forward layers. The generated sample is still rough because this lesson uses a short training run; the larger reference schedule is where a GPU comparison may become useful.
